# ClinicalBridge End-to-End Demonstration

**Team:** Erkan Işık Bacak (2200914), Raymond Lasses (2200274), Ata Uzun (2103247), Kutlay Başar Aklan (2202139)

This annotated notebook demonstrates the two most important orchestration paths:

1. a non-critical alert that triggers parallel EHR and anamnesis processing;
2. a critical alert that bypasses routine synthesis and immediately escalates.

> ClinicalBridge uses entirely simulated data. It is not a diagnostic or clinical decision-making tool.

## 1. Load the reproducible offline system

The repository automatically uses OpenAI when a key is present. For a stable graded demonstration, this notebook explicitly selects offline mode. The same `ClinicalBridge` interface is used in both modes.

In [1]:
import json
from pathlib import Path

from IPython.display import Markdown, display

from clinicalbridge.config import Settings
from clinicalbridge.data_repository import DataRepository
from clinicalbridge.orchestrator import ClinicalBridge
from clinicalbridge.rendering import brief_to_markdown

base = Settings()
settings = Settings(
    project_root=base.project_root,
    openai_api_key="",
    model=base.model,
    embedding_model=base.embedding_model,
    mode="offline",
    rag_backend=base.rag_backend,
    reasoning_effort=base.reasoning_effort,
    retrieval_k=8,
    max_retries=base.max_retries,
)
repository = DataRepository(settings)
system = ClinicalBridge(settings=settings, repository=repository)
print(f"Patients: {repository.patient_count()} | Scenarios: {len(repository.scenarios)} | Mode: offline")

Patients: 12 | Scenarios: 8 | Mode: offline


## 2. Inspect the scenario catalog

The eight scenarios deliberately test different failure modes rather than repeating the same threshold alert.

In [2]:
for scenario in repository.list_scenarios():
    print(
        f"{scenario['scenario_id']}: {scenario['title']} "
        f"({scenario['patient_id']}, expected {scenario['expected_urgency']})"
    )

scenario_01: The Missed Medication (P001, expected Urgent)
scenario_02: The Contextually Benign Alert (P002, expected Routine)
scenario_03: The Silent Deterioration (P003, expected Urgent)
scenario_04: The Incomplete Record (P004, expected Urgent)
scenario_05: The Conflicting Data (P005, expected Urgent)
scenario_06: Critical Hypoxemia (P006, expected Critical)
scenario_07: The Device Artifact (P007, expected Urgent)
scenario_08: Nocturnal Hypertension and NSAID Use (P008, expected Urgent)


## 3. Scenario 03 - Silent deterioration

This scenario tests trend reasoning. No single weight value tells the whole story. The brief must combine a two-week increase with EHR dry weight, heart-failure history, ankle swelling, and mildly worse breathlessness.

In [3]:
scenario_03 = repository.get_scenario("scenario_03")
alert_03 = repository.scenario_alert("scenario_03")
display(Markdown(f"**Scenario:** {scenario_03['description']}"))
print(alert_03.model_dump_json(indent=2))

**Scenario:** Gradual weight gain combines with edema and mild worsening breathlessness.

{
  "alert_id": "A003",
  "patient_id": "P003",
  "timestamp": "2026-05-13T07:05:00Z",
  "device_type": "Connected scale",
  "alert_category": "weight_gain_trend",
  "measurements": {
    "weight": 74.8
  },
  "units": {
    "weight": "kg"
  },
  "baseline": {
    "weight": 72.0
  },
  "thresholds": {
    "fourteen_day_gain": 2.0
  },
  "trend": [
    {
      "days_ago": 14,
      "weight": 72.1
    },
    {
      "days_ago": 7,
      "weight": 73.2
    },
    {
      "days_ago": 3,
      "weight": 74.1
    }
  ],
  "device_metadata": {
    "surface": "hard floor",
    "signal_quality": "good"
  },
  "source_id": "rpm:P003:alert:A003"
}


In [4]:
result_03 = system.run_scenario("scenario_03")
display(Markdown(brief_to_markdown(result_03.brief)))

# Clinical Context Brief

**Patient:** P003  
**Urgency:** Urgent  
**Confidence:** 88%  
**Immediate escalation:** No

## Alert Summary

Weight Gain Trend alert: weight 74.8 kg.

## Patient Snapshot

Active conditions: Heart failure with reduced ejection fraction, Essential hypertension. Current treatment: Furosemide, Carvedilol.

## Contextual Analysis

- The RPM alert was classified Urgent. _Sources: rpm:P003:alert:A003_
- Ankle swelling has gradually increased over six days. _Sources: anam:P003:symptoms:1_
- Mild breathlessness when climbing stairs is slightly worse than usual. _Sources: anam:P003:symptoms:2_
- Reports taking furosemide every morning. _Sources: anam:P003:adherence:1_
- Ate several salty prepared meals this week. _Sources: anam:P003:lifestyle:1_

## Risk Assessment

- Gradual weight increase together with reported ankle swelling may indicate worsening fluid status and deserves prompt clinician assessment. _Sources: rpm:P003:alert:A003, ehr:P003:problem:1, ehr:P003:note:1, anam:P003:symptoms:1, anam:P003:symptoms:2_

## Recommended Actions

- **High:** Review the alert and cited context using the appropriate clinical protocol. — The system provides context only; a clinician must determine next steps. _(confidence 95%; sources: rpm:P003:alert:A003)_

## Uncertainties and Gaps

- The prototype cannot independently verify device accuracy or symptoms.

## Sources

- `rpm:P003:alert:A003` — weight gain trend (rpm_alert)
- `ehr:P003:problem:1` — Heart failure with reduced ejection fraction (ehr_problems)
- `ehr:P003:problem:2` — Essential hypertension (ehr_problems)
- `ehr:P003:med:1` — Furosemide (ehr_medications)
- `ehr:P003:med:2` — Carvedilol (ehr_medications)
- `ehr:P003:lab:1` — BNP (ehr_labs)
- `ehr:P003:lab:2` — Creatinine (ehr_labs)
- `ehr:P003:note:1` — Heart failure monitoring plan (ehr_visit_notes)
- `anam:P003:symptoms:1` — symptoms (anamnesis_symptoms)
- `anam:P003:symptoms:2` — symptoms (anamnesis_symptoms)
- `anam:P003:adherence:1` — adherence (anamnesis_medication_adherence)
- `anam:P003:lifestyle:1` — lifestyle (anamnesis_lifestyle)
- `anam:P003:concern:1` — concern (anamnesis_patient_concerns)

> Educational prototype using simulated data. This output is not a diagnosis or medical advice and must be reviewed by a qualified clinician.


### Inspect the specialized agent outputs

For a non-critical alert, both context agents are present. Their source IDs flow into the final brief.

In [5]:
print("Triage:", result_03.triage.urgency)
print("EHR sources:", len(result_03.ehr_context.citations))
print("Anamnesis sources:", len(result_03.anamnesis_summary.citations))
print("Warnings:", result_03.warnings)
print("Session:", result_03.session_id)

Triage: Urgent
EHR sources: 7
Anamnesis sources: 5
Warnings: []
Session: session-2c20e3e00f22


## 4. Scenario 06 - Critical hypoxemia

This scenario demonstrates the safety exception. Repeated SpO2 of 84% crosses the supplied critical threshold. The orchestrator must not wait for EHR or anamnesis retrieval.

In [6]:
result_06 = system.run_scenario("scenario_06")
display(Markdown(brief_to_markdown(result_06.brief)))
print("EHR agent output:", result_06.ehr_context)
print("Anamnesis agent output:", result_06.anamnesis_summary)
assert result_06.brief.immediate_escalation
assert result_06.ehr_context is None
assert result_06.anamnesis_summary is None

# Clinical Context Brief

**Patient:** P006  
**Urgency:** Critical  
**Confidence:** 99%  
**Immediate escalation:** Yes

## Alert Summary

Critical simulated RPM alert from Pulse oximeter: SpO2 84% is below the critical safety threshold of 88%.

## Patient Snapshot

Full contextual retrieval was intentionally bypassed because the critical safety rule requires immediate human escalation.

## Contextual Analysis

- The deterministic critical threshold was crossed. _Sources: rpm:P006:alert:A006_

## Risk Assessment

- Delay while awaiting automated synthesis could be unsafe; the measurement requires immediate verification and clinician review. _Sources: rpm:P006:alert:A006_

## Recommended Actions

- **Immediate:** Immediately notify the responsible clinician or emergency escalation pathway and verify the reading using the approved clinical protocol. — Critical alerts fail safe and bypass nonessential automation. _(confidence 99%; sources: rpm:P006:alert:A006)_

## Uncertainties and Gaps

- EHR and anamnesis context were not retrieved before escalation.
- Device accuracy and the patient's current symptoms require human verification.

## Sources

- `rpm:P006:alert:A006` — hypoxemia (rpm_alert)

> Educational prototype using simulated data. This output is not a diagnosis or medical advice and must be reviewed by a qualified clinician.


EHR agent output: None
Anamnesis agent output: None


## 5. Review the reproducible evaluation

The automated evaluation checks urgency, retrieval recall, source traceability, safety, latency, key-concern coverage, and a structural unsupported-claim proxy.

In [7]:
evaluation_path = Path("../evaluation/results/offline_v4_evaluation.json")
if not evaluation_path.exists():
    evaluation_path = Path("evaluation/results/offline_v4_evaluation.json")
evaluation = json.loads(evaluation_path.read_text())
evaluation["aggregate"]

{'scenario_count': 8,
 'scenario_pass_rate': 1,
 'triage_accuracy': 1,
 'ehr_retrieval_recall': 1.0,
 'anamnesis_source_recall': 1.0,
 'source_traceability': 1.0,
 'hallucination_proxy_rate': 0.0,
 'key_concern_coverage': 0.7077,
 'safety_compliance': 1,
 'mean_latency_seconds': 0.0025,
 'max_latency_seconds': 0.007}

## 6. Review the completed OpenAI prompt comparison

The same eight scenarios were run live with four prompt iterations. v4 assigns the urgency enum to the deterministic triage tool while retaining the LLM for explanation and retrieval-query generation.

In [8]:
live_results = {}
for version in ("v1", "v2", "v3", "v4"):
    path = Path(f"../evaluation/results/live_{version}_evaluation.json")
    if not path.exists():
        path = Path(f"evaluation/results/live_{version}_evaluation.json")
    live_results[version] = json.loads(path.read_text())["aggregate"]

{
    version: {
        "scenario_pass_rate": metrics["scenario_pass_rate"],
        "triage_accuracy": metrics["triage_accuracy"],
        "mean_latency_seconds": metrics["mean_latency_seconds"],
        "source_traceability": metrics["source_traceability"],
        "safety_compliance": metrics["safety_compliance"],
    }
    for version, metrics in live_results.items()
}

{'v1': {'scenario_pass_rate': 0.875,
  'triage_accuracy': 0.875,
  'mean_latency_seconds': 14.1662,
  'source_traceability': 1.0,
  'safety_compliance': 1},
 'v2': {'scenario_pass_rate': 0.625,
  'triage_accuracy': 0.625,
  'mean_latency_seconds': 15.3123,
  'source_traceability': 1.0,
  'safety_compliance': 1},
 'v3': {'scenario_pass_rate': 0.75,
  'triage_accuracy': 0.75,
  'mean_latency_seconds': 14.5373,
  'source_traceability': 1.0,
  'safety_compliance': 1},
 'v4': {'scenario_pass_rate': 1,
  'triage_accuracy': 1,
  'mean_latency_seconds': 15.6487,
  'source_traceability': 1.0,
  'safety_compliance': 1}}

## 7. Rerun the OpenAI path

Add `OPENAI_API_KEY` to `.env`, leave `CLINICALBRIDGE_MODE=auto`, restart the kernel, and construct `ClinicalBridge()` without overriding settings. The live path uses `gpt-5.4-mini`, OpenAI structured outputs, `text-embedding-3-small`, and persistent Chroma.

Rerun the final live evaluator with:

```bash
python evaluation/evaluate.py --mode live --prompt-version v4
```

The command makes API calls and may incur cost.

## Conclusion

The demonstration shows that ClinicalBridge can coordinate specialized agents, preserve evidence provenance, fail safely on a critical alert, and produce a brief that leaves the final judgment with a clinician. The evaluation does not establish clinical validity; it establishes that the educational prototype behaves according to its implemented contracts on its fictional scenarios.